# Setting

## Import Packages

In [2]:
import pandas as pd
import numpy as np
from scipy import stats
from scipy.stats import chi2_contingency, pointbiserialr

import json
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import math
import itertools
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import Point
import requests
import time
from geopy.geocoders import Nominatim
import time

pd.set_option('display.precision', 4)

# MTA Subway Hourly Ridership
- Instructions:
  1. 2024 Data
     - Download from [MTA-Subway-Hourly-Ridership-2020-2024](https://data.ny.gov/Transportation/MTA-Subway-Hourly-Ridership-2020-2024/wujg-7c2s/explore/query/SELECT%0A%20%20%60transit_timestamp%60%2C%0A%20%20%60borough%60%2C%0A%20%20%60latitude%60%2C%0A%20%20%60longitude%60%2C%0A%20%20sum%28%60ridership%60%29%20AS%20%60sum_ridership%60%0AWHERE%0A%20%20caseless_one_of%28%60borough%60%2C%20%22Manhattan%22%29%0A%20%20AND%20%28%60transit_timestamp%60%0A%20%20%20%20%20%20%20%20%20%3E%3D%20%222024-01-01T00%3A00%3A00%22%20%3A%3A%20floating_timestamp%29%0AGROUP%20BY%20%60transit_timestamp%60%2C%20%60latitude%60%2C%20%60longitude%60%2C%20%60borough%60/page/column_manager)
         - Filters applied:
           1. `borough` = Manhattan
           2. `transit timestamp` >= 2024 Jan 01 12:00:00 AM
         - Grouped by `transit_timestamp`, `latitude`, `longitude`
         - Aggrgation: `SUM(ridership)` as `sum_ridership`
     - Save the exported result as `ridership_2024.csv` into `raw_data/` directory
     
  3. 2025 Data
     - Download from [MTA-Subway-Hourly-Ridership-Beginning-2025](https://data.ny.gov/Transportation/MTA-Subway-Hourly-Ridership-Beginning-2025/5wq4-mkjj/explore/query/SELECT%0A%20%20%60transit_timestamp%60%2C%0A%20%20%60borough%60%2C%0A%20%20%60latitude%60%2C%0A%20%20%60longitude%60%2C%0A%20%20sum%28%60ridership%60%29%20AS%20%60sum_ridership%60%0AWHERE%20caseless_one_of%28%60borough%60%2C%20%22Manhattan%22%29%0AGROUP%20BY%20%60transit_timestamp%60%2C%20%60latitude%60%2C%20%60longitude%60%2C%20%60borough%60%0AHAVING%20%60transit_timestamp%60%20%3C%20%222025-05-01T00%3A00%3A00%22%20%3A%3A%20floating_timestamp%0AORDER%20BY%20%60transit_timestamp%60%20ASC%20NULL%20LAST/page/column_manager)
       - Filters applied:
           1. `borough` = Manhattan
           2. `transit timestamp` < 2025 May 01 12:00:00 AM
         - Grouped by `transit_timestamp`, `latitude`, `longitude`
         - Aggrgation: `SUM(ridership)` as `sum_ridership`
     - Save the exported result as `ridership_20250430.csv` into `raw_data/` directory

In [4]:
# Merge datasets
subway_2024 = pd.read_csv('raw_data/ridership_2024.csv')
subway_2025 = pd.read_csv('raw_data/ridership_20250430.csv')
subway_df = pd.concat([subway_2024, subway_2025])

# Convert transit_timestamp column to datetime type 
subway_df['transit_timestamp'] = pd.to_datetime(subway_df['transit_timestamp'])
subway_df = subway_df.sort_values('transit_timestamp').reset_index(drop=True)

# Assume that each passenger influences the station 1 hour before and during the entrance time
influence_df = subway_df.copy()
influence_df['transit_timestamp'] = influence_df['transit_timestamp'] - pd.Timedelta(hours=1)

subway_with_influence_df = pd.concat([subway_df, influence_df])
aggregated_subway_df = subway_with_influence_df.groupby(['transit_timestamp', 'borough', 'latitude', 'longitude'])['ridership'].sum().reset_index()

aggregated_subway_df = aggregated_subway_df[aggregated_subway_df['transit_timestamp'] >= pd.to_datetime('2024-01-01')]

# The borough column is used to ensure the downloaded dataset is Manhattan only, no need to saved to CSV file
aggregated_subway_df = aggregated_subway_df[['transit_timestamp', 'latitude', 'longitude', 'ridership']]

aggregated_subway_df.to_csv('mta_subway_20240101_20250430.csv', index=False)